### модель paging (как понять, что страниц больше нет)

Окей, надо уточнить вопрос "Как понять что страниц больше нет".
В документации явного ответа на этот вопрос нет, потому выясним опытным путем:

- составим запрос к api для этого возьмем запросы
  - Информация о вакансиях в конкретном регионе:
    - <http://opendata.trudvsem.ru/api/v1/vacancies/region/%region_code%
  - число элементов (limit) и смещение (offset)
    - <http://opendata.trudvsem.ru/api/v1/vacancies?offset=1&limit=100>
- получим данные по запросу с указанием региона лимитом и смещением, <http://opendata.trudvsem.ru/api/v1/vacancies?offset=1&limit=100>
  - Как делал смотрите ниже в блокноте
  - **Вывод** Как можно увидеть расчеты нам не помогли. Вообщем странное поведение API, конечно, но ладно. Пока оставим именно так: качаем с 1-ой страницы и до 500-ки с несколькими ретраями, затем прекращаем.
- Проверим еще коды регионов. Гипотеза такова, что апи использует стандартные регионы. Потому:
  - Находим источник со справочником регионов, которые можно подгружать
  - Грузим-проверяем-делаем вывод
  - Коды регионов валидны, берем 3 шт:
    - Москва: 77
    - Питер: 78
    - Свердлов. обл.: 66
    - **Итого:**
    - [{'region_66': 18078}, {'region_77': 15556}, {'region_78': 14227}]


In [10]:
from requests import get
from json import loads, load, dump

Тк это госапи, то предположу что коды регионов общепринятые. Те Сврдл. обл это 66 регион. Но это нужно проверить дополнительно


In [18]:
region_code_ural = 66
url_api = f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}"

loads(get(url_api).content)

{'status': '200',
 'request': {'api': 'v1'},
 'meta': {'total': 18125, 'limit': 100},
 'results': {'vacancies': [{'vacancy': {'id': '5a5c4f38-0bf8-11f1-babf-a582caf9364c',
     'source': 'Работодатель',
     'region': {'region_code': '6600000000000',
      'name': 'Свердловская область'},
     'company': {'companycode': '1026600931675',
      'email': 'kulz@kulz.ru',
      'hr-agency': False,
      'inn': '6666000100',
      'kpp': '661201001',
      'name': 'АО "КУЛЗ"',
      'ogrn': '1026600931675',
      'url': 'https://trudvsem.ru/company/1026600931675'},
     'creation-date': '2026-02-17',
     'date_modify': '2026-02-18T13:36:37+0300',
     'salary': 'от 35000',
     'salary_min': 35000,
     'salary_max': 45000,
     'job-name': 'Пирометрист',
     'vac_url': 'https://trudvsem.ru/vacancy/card/1026600931675/5a5c4f38-0bf8-11f1-babf-a582caf9364c',
     'employment': 'Полная занятость',
     'schedule': 'Полный рабочий день',
     'qualification': '3 разряд',
     'requirements': 'О

Хорошо, видим вот такую основную структуру ответа api на запрос: <http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}>

```json
{
  "status": "200",
  "request": { "api": "v1" },
  "meta": { "total": 18125, "limit": 100 },
  "results": { "vacancies": ["...", "..."] }
}
```

Нас интересует meta,в которой описано:

- total: итого, вакансий по запросу
- limit: сколько вернули по странице

Версионности, ятак понимаю у api нет, те обход 18125/100 ~ 182 страниц займет время, а это означает что... А нет, ничего это не означет, свежие вакансии, появившиеся за время обхода будут в конце (**вероятно**, но это не точно. Надо проверить)

Выходит, что при запросах каждой страницы -- надо каждый раз проверять <meta.total> в запросе, и корректировать кол-во страниц (**не это пока не точно**) путем вычисления total/limit + округление в большую сторону. Хорошо, тогда проведем эксперимент:

- получим данные по запросу с указанием региона лимитом и смещением, <http://opendata.trudvsem.ru/api/v1/vacancies?offset=1&limit=100>:
  - 1: limit=10, offset=1
  - 2: limit=10, offset= (total_vacancies - 1) // limit + 1
    - округл в большую сторону: (total_vacancies - 1) // limit + 1
  - 3: limit=10, offset= (total_vacancies - 1) // limit + 2
  - если в 1, 2 будет ответ, а в 3 не будет, то это означает что мы "нащупали дно"


In [60]:
# limit=10, offset=1

region_code_ural = 66
limit = 10
offset = 1

url_api = f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset}&limit={limit}"

content_limit10_offset1 = loads(get(url_api).content)
total_vacancies = content_limit10_offset1["meta"]["total"]

# вычислим последнюю страничку
offset_last_page = (total_vacancies - 1) // limit + 1

url_api = f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset_last_page}&limit={limit}"

# проверяем есть контект на странице вычисленной
content_limit10_offset_last_page = loads(get(url_api).content)

print(offset_last_page)
content_limit10_offset_last_page

1813


{'status': '500',
 'request': {'api': 'v1'},
 'meta': {'error': 'При выполнении запроса произошла ошибка. Проверьте синтаксис запроса и\\или обратитесь в службу поддержки'}}

Но почему-то приходит 500-ка. На всякий случай проверю 2-ю и, скажем 200 страницу, ну и 1


In [68]:
offset_pages = [2, 200, 450, 700, 1000, 1813]

contents_page = [
    {
        f"offset_page_{offset_page}": get(
            f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset_page}&limit={limit}"
        ).status_code
    }
    for offset_page in offset_pages
]

contents_page

[{'offset_page_2': 200},
 {'offset_page_200': 200},
 {'offset_page_450': 200},
 {'offset_page_700': 200},
 {'offset_page_1000': 500},
 {'offset_page_1813': 500}]

Как можно увидеть расчеты нам не помогли. Вообщем странное поведение API, конечно, но ладно. Пока оставим именно так: качаем с 1-ой страницы и до 500-ки с несколькими ретраями, затем прекращаем. Еще попробуем по 100 вакансий, те макс кол-во. Возможно тут математика сойдется


In [72]:
# limit=100, offset=1

region_code_ural = 66
limit = 100
offset = 1

url_api = f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset}&limit={limit}"

content_limit100_offset1 = loads(get(url_api).content)
total_vacancies = content_limit10_offset1["meta"]["total"]

# вычислим последнюю страничку
offset_last_page = (total_vacancies - 1) // limit + 1

url_api = f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset_last_page}&limit={limit}"

# проверяем есть контект на странице вычисленной
content_limit100_offset_last_page = loads(get(url_api).content)

print(offset_last_page)
content_limit100_offset_last_page

182


{'status': '500',
 'request': {'api': 'v1'},
 'meta': {'error': 'При выполнении запроса произошла ошибка. Проверьте синтаксис запроса и\\или обратитесь в службу поддержки'}}

In [75]:
offset_last_page

182

In [73]:
region_code_ural = 66
limit = 100
offset = 1

offset_pages = [2, 25, 50, 100, 160, 182]

contents_page = [
    {
        f"offset_page_{offset_page}": get(
            f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code_ural}?offset={offset_page}&limit={limit}"
        ).status_code
    }
    for offset_page in offset_pages
]

contents_page

[{'offset_page_2': 200},
 {'offset_page_25': 200},
 {'offset_page_50': 200},
 {'offset_page_100': 500},
 {'offset_page_160': 500},
 {'offset_page_182': 500}]

Вообщем закономерность есть, но математика по прежнему не рабоатает. Оставим на потом, когда будем проверять численность полученных json


Теперь проверим регионы:

Гипотеза: API использует стандартный справочный регионов России. Проверим:

- По хорошему надо взять подгружаемый справочник регионов из открытого источника [ФИАС](https://fias-public-service.nalog.ru/api/spas/v2.0/swagger/index.html), однако надо делать доступ, потому в демонстрационных целях воспользуемся обычным json файлом, подготовленным вручную с [гита](https://github.com/arbaev/russia-cities/blob/master/russia-regions.json)
- пройдемся по каждому региону, взяв по 1-ой вакансии


In [103]:
# сдампим байты с файлика с гита

with open("files/russia-regions.json", "r") as f:
    regions_code = [region_info.get("code") for region_info in load(f)]
    f.close()

# по каждому из кодов закинем запрос в api, под одной из вакансий


limit = 1
offset = 1

# соберем статусы по всем регионам. Если список будет пустой, значит регионы валидны

contents_page = [
    {f"region_{region_code}_status": status_code}
    for region_code in regions_code
    if (
        status_code := get(
            f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code}?limit={limit}"
        ).status_code
        != 200
    )
]

contents_page

[]

Штош, регионы все валидны, по каждому получен код 200 в ответе, но для нашего стенда возьмем данные трех регионов:

- Москва
- Питер
- Свердловская обл.


In [15]:
# Глянем сразу сколько вакансий доступно на момент исследования по 3-м регионам

# Сформируем справочник выбраных городов, названия которых я конечно же скопировал из json

regions = ["Москва", "Санкт-Петербург", "Свердловская"]

with open("files/russia-regions.json", "r") as f:
    regions_code = [
        region_info.get("code")
        for region_info in load(f)
        if region_info.get("name") in regions
    ]
    f.close()

# соберем кол-во вакансий по выбранным регионам. Справочник сделаем на основе существующего файлика:
limit = 2
offset_page = 1

contents_page = [
    {
        f"region_{region_code}": loads(
            get(
                f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code}?offset={offset_page}&limit={limit}"
            ).content
        )["meta"]["total"]
    }
    for region_code in regions_code
]
contents_page

[{'region_66': 17973}, {'region_77': 15601}, {'region_78': 14118}]

Соберем реальные примеры в файл

In [17]:
contents_page = [
    {
        f"region_{region_code}": loads(
            get(
                f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{region_code}?offset={offset_page}&limit={limit}"
            ).content
        )
    }
    for region_code in regions_code
]

with open("3.1.1.1_example.json", mode="w") as f:
    dump(contents_page, f, ensure_ascii=False)
    f.close()


Списком подать регионы нельзя, только по одному:

In [18]:
loads(
    get(
        f"http://opendata.trudvsem.ru/api/v1/vacancies/region/{regions_code}?offset={offset_page}&limit={limit}"
    ).content
)

{'status': '200',
 'request': {'api': 'v1'},
 'meta': {'total': 0, 'limit': 2},
 'results': {}}